# MobileNetV2 Diagnostic & Interpretability Analysis: External Dataset
---
Full diagnostic suite covering global accuracy, confusion matrix, analytical Grad-CAM, latent space projections, and confidence calibration on the External Merged 35-make dataset.


## 1. Setup & Context Initialization


In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Import the diagnostic engine
import output_analysis_external as oae

print(f"Context Model: {oae.context.model_path}")
print(f"Classes: {oae.context.num_classes}")


## 2. Evaluate Test Split & Summary Metrics


In [ ]:
test_csv = oae.context.test_csv_path
if test_csv and Path(test_csv).exists():
    df_test = pd.read_csv(test_csv)
    print(f"Test samples: {len(df_test):,}")
    
    model = oae.MobileNetV2Evaluator(num_classes=oae.context.num_classes, checkpoint_path=oae.context.model_path)
    y_true, y_pred, y_probs, embeddings = oae.evaluate_test_dataset(model, df_test, batch_size=64)
    metrics = oae.compute_and_save_metrics(y_true, y_pred, y_probs, oae.context.output_dir)
else:
    print("Test CSV not found. Please verify context configuration.")


## 3. Confusion Matrix & Misclassification Patterns


In [ ]:
from IPython.display import Image, display
cm_img = oae.context.plots_dir / "confusion_matrix.png"
if cm_img.exists():
    display(Image(str(cm_img)))
else:
    print("Confusion matrix plot not available yet.")


## 4. Latent Space Embedding Analysis (t-SNE & PCA)


In [ ]:
tsne_img = oae.context.plots_dir / "latent_space_tsne_pca.png"
if tsne_img.exists():
    display(Image(str(tsne_img)))
else:
    print("Latent space plot not available yet.")


## 5. Grad-CAM (Class Activation Maps)
Random test images with the CAM of the predicted class, computed from the ONNX `class_maps` output.


In [ ]:
import os

col = "image_path" if "image_path" in df_test.columns else "image_rel_path"
sample = df_test.sample(n=min(6, len(df_test)), random_state=0)

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
for ax in axes.ravel():
    ax.axis("off")
for ax, (_, row) in zip(axes.ravel(), sample.iterrows()):
    p = row[col]
    if oae.context.base_img_dir and not os.path.isabs(p):
        p = str(Path(oae.context.base_img_dir) / p)
    tensor, img = oae.load_and_preprocess_image(p, oae.context.img_size, oae.context.crop_top_pct, oae.context.crop_bottom_pct)
    pred = int(model(tensor).argmax(dim=1).item())
    if model.class_maps is None:
        print("Model has no 'class_maps' output; re-export it with the current training script.")
        break
    cam = oae.compute_gradcam(model, tensor, pred)
    ax.imshow(oae.overlay_cam_on_image(img, cam))
    ax.set_title(f"true: {oae.context.idx_to_class[int(row['label'])]}\npred: {oae.context.idx_to_class[pred]}", fontsize=10)
plt.tight_layout()
plt.show()


## 6. Confidence Calibration & Rejection Curves (tau = 0.70)


In [ ]:
calib_img = oae.context.plots_dir / "calibration_curve.png"
if calib_img.exists():
    display(Image(str(calib_img)))
else:
    print("Calibration plot not available yet.")
